In [1]:
!pip install -q langchain langchain-community langchain-huggingface chromadb pypdf sentence-transformers "transformers<5.0.0" torch

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [2]:
university_text = """
University of Plymouth - Computer Science Handbook

1. Course Overview: The BSc Computer Science program is designed to produce competent computer professionals.
2. Module PUSL3205: Large Language Models and Generative AI is a 20-credit module assessed via 100% coursework.
3. Assessment Deadlines: The C1 written assignment and P1 practical portfolio are due on 3/12/2026 at 16:00 SL time.
4. Lab Requirements: Students must complete labs on Machine Learning, Transformers, Fine-Tuning, RAG, and AI Agents.
5. Contact: The module leader is Ms. Thilini Bakmeedeniya.
"""

with open("university_handbook.txt", "w") as f:
    f.write(university_text)

print("✅ Document created successfully!")

✅ Document created successfully!


In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("1. Loading document...")
loader = TextLoader("university_handbook.txt")
documents = loader.load()

print("2. Splitting document into chunks...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
chunks = text_splitter.split_documents(documents)

print("3. Creating Embeddings and storing in ChromaDB...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings)

print(f"✅ RAG Database ready with {len(chunks)} chunks!")

/tmp/ipykernel_641/2916813727.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


1. Loading document...
2. Splitting document into chunks...
3. Creating Embeddings and storing in ChromaDB...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

... (5 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ RAG Database ready with 10 chunks!


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print("Loading the AI model (Flan-T5)...")
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def ask_ai(question):
    # 1. Search the database for the 2 most relevant chunks
    docs = vectorstore.similarity_search(question, k=2)
    context = "\n".join([doc.page_content for doc in docs])

    # 2. Give the chunks to the AI
    prompt = f"Answer the question based on this text:\n{context}\n\nQuestion: {question}"

    # 3. Generate the answer
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=200)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

print("\n--- Test 1 (Baseline) ---")
q1 = "What is the module PUSL3205 about?"
print(f"Q: {q1}")
print(f"A: {ask_ai(q1)}")

Loading the AI model (Flan-T5)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


--- Test 1 (Baseline) ---
Q: What is the module PUSL3205 about?
A: Large Language Models and Generative AI


In [5]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
import os

print("Please click 'Choose Files' to upload a PDF...")
uploaded = files.upload()

if uploaded:
    # Get the name of the uploaded file
    filename = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {filename}")

    print("Reading and processing your PDF...")
    # Save the uploaded bytes to a temporary PDF file
    with open("temp_uploaded.pdf", "wb") as f:
        f.write(uploaded[filename])

    # Load the PDF
    loader = PyPDFLoader("temp_uploaded.pdf")
    pdf_docs = loader.load()

    # Split and embed the new PDF
    pdf_chunks = text_splitter.split_documents(pdf_docs)
    vectorstore.add_documents(pdf_chunks)

    print(f"✅ Added {len(pdf_chunks)} new chunks from your PDF to the database!")

    # Ask a question about the new PDF
    test_q = "What is this document about?"
    print(f"\nQ: {test_q}")
    print(f"A: {ask_ai(test_q)}")

    # Clean up the temp file
    os.remove("temp_uploaded.pdf")

Please click 'Choose Files' to upload a PDF...


Saving Lab sheet 5.pdf to Lab sheet 5.pdf

✅ Uploaded: Lab sheet 5.pdf
Reading and processing your PDF...
✅ Added 57 new chunks from your PDF to the database!

Q: What is this document about?
A: Science/Tech
